# EcoOrbit in Google Colab — interactive results website

Upload `EcoOrbit_813_Submission_Starter.zip` through the Files sidebar to `/content`, run extraction and installation, then build and download the self-contained HTML website. Its Solution page maps five NASA EMIT V002 CH4 plume-complex catalog geometries near Jeddah and Seropedica. These are not source-attributed annual emissions. Optional full NASA Earthdata downloads require your own login and large disk space.


In [ ]:
from pathlib import Path
import zipfile
archive = Path('/content/EcoOrbit_813_Submission_Starter.zip')
assert archive.is_file(), 'Upload the ZIP using the Colab Files sidebar before running this cell.'
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None, 'The ZIP is incomplete; upload it again.'
    z.extractall('/content')
project = Path('/content/EcoOrbit_v2')
assert (project / 'dashboard.py').exists()
print('Project ready:', project)


In [ ]:
%cd /content/EcoOrbit_v2
%pip -q install -r requirements.txt


## Build and download the results website

This generates one self-contained HTML file from the bundled real result files. The charts, filters, site map and methodology work after you open it locally; no Colab port or localhost is involved. The HTML file is also included in the ZIP.


In [ ]:
from src.build_website import build
from google.colab import files
website_path = build()
print(f'Website ready: {website_path} ({website_path.stat().st_size / 1e6:.1f} MB)')
files.download(str(website_path))


## Inspect the included real plume and pollutant records

The source URLs and image links are retained. Rates reported in the articles are instantaneous or preliminary source estimates; they are not annual carbon inventories.


In [ ]:
import pandas as pd
display(pd.read_csv('data/published_plumes.csv')[['plume_id','site_name','gas','scene_timestamp','reported_rate_kg_h','rate_basis','source_url']])
display(pd.read_csv('results/real/methane/observed_us_regional_ch4.csv'))


## Independent NASA landfill methane findings

The five NASA EMIT V002 CH4 plume-complex catalog geometries are real results near Jeddah and Seropedica. The plotted coordinates are approximate landfill points; inspect full products before attribution. No EMIT CO2 plume was found in the two 2024-2025 catalog searches.


In [ ]:
from IPython.display import Image, display
near = pd.read_csv("results/real/landfill_gases/emit_search.csv")
display(near[["site_id","gas","granule_id","plume_distance_km","status"]])
display(Image(filename="results/real/landfill_gases/emit_landfill_footprints.png"))


## Interactive dashboard in the notebook

Use the **View**, **City**, **Gas**, and **Column** menus below. The results are bundled observed satellite products and cited published plumes; they do not attribute emissions to a factory. If widgets are blocked, use the direct `show_section` example below.


In [ ]:
import subprocess, sys
%pip -q install ipywidgets
from google.colab import output
output.enable_custom_widget_manager()
from src.colab_dashboard import show_dashboard, show_section
show_dashboard()
# Fallback if the interactive controls are hidden: show_section('Heat')


## Optional: download case-specific Sentinel-5P methane

Each command can download large orbital NetCDF files. Start with one case and one scene. A valid regional XCH4 difference is not a facility plume, and no scene is not a non-detection. Rerun the dashboard after the command.


In [ ]:
# Uncomment ONE when you are ready for a potentially large download:
# !python -m src.methane_cases --site jeddah_landfill --max-scenes 1
# !python -m src.methane_cases --site hassi_rmel --max-scenes 1
# !python -m src.methane_cases --site stanford_arizona --max-scenes 1


## Optional: import new Carbon Mapper CH4 and CO2 plume exports

Export a real CSV or GeoJSON plume list from the Carbon Mapper portal, upload it below, and reopen the dashboard. Spatial proximity in the output requires independent source review.


In [ ]:
# Uncomment to upload a portal export through the Colab Files sidebar,
# then change the filename and run this cell:
# plume_file = '/content/plumes.csv'
# subprocess.run([sys.executable, '-m', 'src.import_methane_plumes', plume_file], cwd=str(project), check=True)


## Optional: save results to Google Drive

Colab removes local files when the runtime ends. The source cache can be large, so copy only the resulting CSVs or ZIP unless you have Drive space.


In [ ]:
# Uncomment to keep the output files:
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copytree(project / 'results', '/content/drive/MyDrive/EcoOrbit_results', dirs_exist_ok=True)


## Previous analysis: factory hyperspectral classifier

The included 48 real Tanager site feature rows can reproduce the all-city site split **without downloading the 3.3 GB raw scenes**. It writes model, split, predictions, accuracy and chart to `results/real/classifier/`. Source code: `src/evaluate_sites.py` and `src/train_classifier.py`. The measured test uses factory versus school controls; green image patches remain an independent reference, not training labels.


In [ ]:
from src.evaluate_sites import run as evaluate_classifier
evaluate_classifier()
import json
metrics = json.loads(Path('results/real/classifier/all_city_test_metrics.json').read_text())
print('Held-out sites:', metrics['n_test'], 'accuracy:', round(metrics['test_accuracy'],3), 'AUC:', round(metrics['test_auc'],3))
display(pd.read_csv('results/real/classifier/all_city_test_predictions.csv')[['city','name','label','factory_screen_score','predicted_label']])


## Previous analysis: Landsat heat

The bundled `site_heat.csv` is real, QA-filtered Landsat land-surface temperature near the factory points and their surrounding reference rings. Source code: `src/site_heat.py`. Display the existing output first. Reprocessing is optional and requires internet access to Planetary Computer.


In [ ]:
heat = pd.read_csv('results/real/heat/site_heat.csv')
display(heat[['city','name','date','surface_c','reference_c','surface_excess_c']].head(20))
print('QA-valid site-date rows:', len(heat))


## Previous analysis: Sentinel-5P NO₂ and SO₂

The bundled output has QA-filtered atmospheric column values and explicit missing statuses. It is a coarse column comparison, not a facility emission rate. Source code: `src/site_pollutants.py`. The dashboard's **NO₂, SO₂ and aerosols** tab filters these observations.


In [ ]:
gases = pd.read_csv('results/real/pollutants/site_no2_so2.csv')
for gas in ['no2','so2']:
    print('\n', gas.upper())
    display(gases[gases['product'].eq(gas)][['city','name','date','column_mol_m2','nearest_pixel_center_km','qa','status']].head(12))


## Optional: refresh satellite scenes

These commands use **real remote satellite scenes** and can take time or several GB. Start with a single city/product. Do not run them merely to view the included outputs. The Tanager download and extraction are required only for adding new training sites or reproducing raw spectral extraction.


In [ ]:
# Uncomment individual commands when you want to reprocess from source:
# subprocess.run([sys.executable, '-m', 'src.site_pollutants', '--city', 'Detroit', '--product', 'no2', '--max-days', '1'], cwd=str(project), check=True)
# subprocess.run([sys.executable, '-m', 'src.site_pollutants', '--city', 'Detroit', '--product', 'so2', '--max-days', '1'], cwd=str(project), check=True)
# subprocess.run([sys.executable, '-m', 'src.site_heat'], cwd=str(project), check=True)
# subprocess.run([sys.executable, '-m', 'src.multisite_download'], cwd=str(project), check=True)
# subprocess.run([sys.executable, '-m', 'src.train_classifier'], cwd=str(project), check=True)


## Actual Jeddah and Seropédica EMIT pixels

Real Jeddah source crops and both Brazil plume rasters are bundled. This computes pixel enhancement and mass plus unvalidated diagnostic flux scenarios.


In [1]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pyproj==3.7.2'], check=True)
subprocess.run([sys.executable, '-m', 'src.emit_cases'], cwd=str(project), check=True)
subprocess.run([sys.executable, '-m', 'src.build_website'], cwd=str(project), check=True)
files.download(str(project / 'EcoOrbit_Results_Website.html'))
